# 00 · Exploración de datos

**Cloud Provider Analytics · Big Data (ITBA) · Primera entrega**

Lectura y perfil inicial de las ocho fuentes de `datalake/landing/`. Landing no se modifica: el notebook solo lee.
Los resultados respaldan la sección 3 del documento de diseño.

## 1. Configuración

En Colab, montar Drive y apuntar `LANDING` a la carpeta del dataset.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
LANDING = '/content/drive/MyDrive/cloud_provider_challenge_dataset_v1/datalake/landing'

Mounted at /content/drive


In [ ]:
import os, glob, json
import pandas as pd

LANDING = os.environ.get('LANDING', '/content/drive/MyDrive/cloud_provider_challenge_dataset_v1/datalake/landing')
pd.set_option('display.width', 160)
print('Landing:', LANDING)
print('Archivos CSV:', sorted(os.path.basename(p) for p in glob.glob(os.path.join(LANDING, '*.csv'))))
print('Archivos JSONL:', len(glob.glob(os.path.join(LANDING, 'usage_events_stream', '*.jsonl'))))

Landing: /content/drive/MyDrive/cloud_provider_challenge_dataset_v1/datalake/landing
Archivos CSV: ['billing_monthly.csv', 'customers_orgs.csv', 'marketing_touches.csv', 'nps_surveys.csv', 'resources.csv', 'support_tickets.csv', 'users.csv']
Archivos JSONL: 120


## 2. Master Data (CSV)

Filas, columnas, vacíos y duplicados en la clave natural de cada archivo.

In [ ]:
KEYS = {'customers_orgs.csv': 'org_id', 'users.csv': 'user_id', 'resources.csv': 'resource_id',
        'support_tickets.csv': 'ticket_id', 'marketing_touches.csv': 'touch_id',
        'nps_surveys.csv': None, 'billing_monthly.csv': 'invoice_id'}
md_data = {}
rows = []
for name, key in KEYS.items():
    path = os.path.join(LANDING, name)
    raw = pd.read_csv(path, dtype=str, keep_default_na=False)
    md_data[name] = pd.read_csv(path)
    rows.append({'archivo': name, 'filas': len(raw), 'columnas': raw.shape[1],
                 'celdas_vacias': int((raw == '').sum().sum()),
                 'duplicados_clave': int(raw[key].duplicated().sum()) if key else '—'})
resumen = pd.DataFrame(rows)
print(resumen.to_string(index=False))
print('\nTotal de filas de Master Data:', resumen['filas'].sum())

              archivo  filas  columnas  celdas_vacias duplicados_clave
   customers_orgs.csv     80        11             11                0
            users.csv    800         7            139                0
        resources.csv    400         7             83                0
  support_tickets.csv   1000         8            494                0
marketing_touches.csv   1500         7              0                0
      nps_surveys.csv     92         4             29                —
  billing_monthly.csv    240         8            137                0

Total de filas de Master Data: 4112


## 3. Hallazgos de calidad en la Master Data

In [ ]:
c, u, t = md_data['customers_orgs.csv'], md_data['users.csv'], md_data['support_tickets.csv']
n, b, m = md_data['nps_surveys.csv'], md_data['billing_monthly.csv'], md_data['marketing_touches.csv']

hallazgos = [
    ('customers_orgs', 'nps_score nulo', c.nps_score.isna().sum()),
    ('customers_orgs', 'nps_score fuera de [-100, 100]', ((c.nps_score < -100) | (c.nps_score > 100)).sum()),
    ('customers_orgs', 'plan enterprise con is_enterprise = False', ((c.plan_tier == 'enterprise') & (~c.is_enterprise)).sum()),
    ('users', 'last_login nulo', u.last_login.isna().sum()),
    ('users', 'last_login anterior a created_at', (pd.to_datetime(u.last_login) < pd.to_datetime(u.created_at)).sum()),
    ('support_tickets', 'tickets abiertos (resolved_at nulo)', t.resolved_at.isna().sum()),
    ('support_tickets', 'csat fuera de 1–5', ((t.csat < 1) | (t.csat > 5)).sum()),
    ('support_tickets', 'severidad critical', (t.severity == 'critical').sum()),
    ('nps_surveys', 'organizaciones con dos encuestas', (n.org_id.value_counts() == 2).sum()),
    ('billing_monthly', 'credits nulo', b.credits.isna().sum()),
    ('billing_monthly', 'subtotal negativo', (b.subtotal < 0).sum()),
    ('billing_monthly', 'organizaciones con más de una moneda', (b.groupby('org_id').currency.nunique() > 1).sum()),
    ('marketing_touches', 'converted sin clicked', (m.converted & ~m.clicked).sum()),
]
print(pd.DataFrame(hallazgos, columns=['fuente', 'hallazgo', 'casos']).to_string(index=False))

           fuente                                  hallazgo  casos
   customers_orgs                            nps_score nulo     11
   customers_orgs            nps_score fuera de [-100, 100]      1
   customers_orgs plan enterprise con is_enterprise = False      9
            users                           last_login nulo    139
            users          last_login anterior a created_at    232
  support_tickets       tickets abiertos (resolved_at nulo)    240
  support_tickets                         csat fuera de 1–5     40
  support_tickets                        severidad critical     56
      nps_surveys          organizaciones con dos encuestas     32
  billing_monthly                              credits nulo    137
  billing_monthly                         subtotal negativo     13
  billing_monthly      organizaciones con más de una moneda     50
marketing_touches                     converted sin clicked     96


In [ ]:
print('Tipo de cambio por moneda (billing_monthly):')
print(b.groupby('currency').exchange_rate_to_usd.agg(['count', 'min', 'max']).to_string())
orgs = set(c.org_id)
print('\norg_id huérfanos por fuente:')
for name in ['users.csv', 'resources.csv', 'support_tickets.csv', 'marketing_touches.csv', 'nps_surveys.csv', 'billing_monthly.csv']:
    print(f'  {name:24s}', int((~md_data[name].org_id.isin(orgs)).sum()))

Tipo de cambio por moneda (billing_monthly):
          count      min      max
currency                         
ARS          51  0.00133  0.00162
EUR          29  0.99810  1.19808
USD         160  0.85463  1.11791

org_id huérfanos por fuente:
  users.csv                0
  resources.csv            0
  support_tickets.csv      0
  marketing_touches.csv    0
  nps_surveys.csv          0
  billing_monthly.csv      0


## 4. Eventos de uso (JSONL)

Se leen línea por línea para detectar JSON inválidos y valores de `value` que llegan como texto.

In [ ]:
files = sorted(glob.glob(os.path.join(LANDING, 'usage_events_stream', '*.jsonl')))
records, invalid = [], 0
for path in files:
    with open(path, encoding='utf-8') as fh:
        for line in fh:
            if not line.strip():
                continue
            try:
                d = json.loads(line)
            except json.JSONDecodeError:
                invalid += 1
                continue
            d['value_is_text'] = isinstance(d.get('value'), str)
            d['source_file'] = os.path.basename(path)
            records.append(d)
ev = pd.DataFrame(records)
ev['event_ts'] = pd.to_datetime(ev.timestamp, utc=True, errors='coerce')
print(f'Archivos: {len(files)} · eventos: {len(ev)} · JSON inválidos: {invalid}')
print('Rango temporal:', ev.event_ts.min(), '→', ev.event_ts.max(), f'({ev.event_ts.dt.date.nunique()} días)')
print('Servicios:', ev.service.value_counts().to_dict())

Archivos: 120 · eventos: 43200 · JSON inválidos: 0
Rango temporal: 2025-07-03 00:02:00+00:00 → 2025-08-31 23:58:00+00:00 (60 días)
Servicios: {'compute': 12498, 'storage': 7614, 'database': 7419, 'networking': 6921, 'analytics': 4590, 'genai': 4158}


### 4.1 Evolución de esquema

In [ ]:
print('Eventos por schema_version:', ev.schema_version.value_counts().to_dict())
print('Último evento v1:', ev[ev.schema_version == 1].event_ts.max())
print('Primer evento v2:', ev[ev.schema_version == 2].event_ts.min())
print('carbon_kg informado en v1:', int(ev[ev.schema_version == 1].carbon_kg.notna().sum()))
print('genai_tokens fuera del servicio genai:', int(ev[ev.service != 'genai'].genai_tokens.notna().sum()))

Eventos por schema_version: {2: 32400, 1: 10800}
Último evento v1: 2025-07-17 23:56:00+00:00
Primer evento v2: 2025-07-18 00:01:00+00:00
carbon_kg informado en v1: 0
genai_tokens fuera del servicio genai: 0


### 4.2 Campos `value` y `unit`

In [ ]:
value_num = pd.to_numeric(ev.value.astype(str).where(ev.value.notna()), errors='coerce')
print('value como texto:', int(ev.value_is_text.sum()))
print('value nulo:', int(ev.value.isna().sum()))
print('value texto no casteable:', int((ev.value.notna() & value_num.isna()).sum()))
print('unit nulo con value presente:', int((ev.value.notna() & ev.unit.isna()).sum()))
print('\nRelación metric → unit (permite imputar la unidad):')
print(pd.crosstab(ev.metric, ev.unit.fillna('<NULL>')).to_string())

value como texto: 1309
value nulo: 877
value texto no casteable: 0
unit nulo con value presente: 2038

Relación metric → unit (permite imputar la unidad):
unit              <NULL>  count  gb_hours  hours
metric                                          
cpu_hours            505      0         0  10187
requests             916  18596         0      0
storage_gb_hours     654      0     12342      0


### 4.3 Costos y anomalías

In [ ]:
cost = ev.cost_usd_increment
p99_servicio = ev.groupby('service').cost_usd_increment.transform(lambda s: s.quantile(0.99))
print('Mediana de costo por evento (USD):', round(cost.median(), 4), '· máximo:', cost.max())
print('Costos < -0,01 (regla R2):', int((cost < -0.01).sum()))
print('Costos entre -0,01 y 0 (tolerancia):', int(((cost >= -0.01) & (cost < 0)).sum()))
print('Picos > 3 × p99 del servicio (flag):', int((cost > 3 * p99_servicio).sum()))
print('event_id nulos:', int(ev.event_id.isna().sum()), '· duplicados:', int(ev.event_id.duplicated().sum()))

Mediana de costo por evento (USD): 1.0044 · máximo: 317.4308
Costos < -0,01 (regla R2): 211
Costos entre -0,01 y 0 (tolerancia): 5
Picos > 3 × p99 del servicio (flag): 79
event_id nulos: 0 · duplicados: 0


### 4.4 Orden de llegada de los archivos

Si cada archivo trae eventos de todo el período, el orden de llegada no sigue el tiempo de evento y el watermark debe contemplarlo.

In [ ]:
por_archivo = ev.groupby('source_file').event_ts.agg(['min', 'max'])
completos = ((por_archivo['min'].dt.date == ev.event_ts.min().date()) &
             (por_archivo['max'].dt.date == ev.event_ts.max().date())).sum()
print(f'Archivos que cubren los 60 días completos: {completos} de {len(por_archivo)}')
print(por_archivo.head(3).to_string())

Archivos que cubren los 60 días completos: 120 de 120
                                             min                       max
source_file                                                               
events_part_0000.jsonl 2025-07-03 00:08:00+00:00 2025-08-31 16:32:00+00:00
events_part_0001.jsonl 2025-07-03 02:38:00+00:00 2025-08-31 23:51:00+00:00
events_part_0002.jsonl 2025-07-03 14:16:00+00:00 2025-08-31 20:47:00+00:00


## 5. Integridad y tamaño esperado del primer mart

In [ ]:
res = md_data['resources.csv']
print('resource_id sin match en resources.csv:', int((~ev.resource_id.isin(res.resource_id)).sum()))
print('org_id sin match en customers_orgs.csv:', int((~ev.org_id.isin(c.org_id)).sum()))
validos = ev[cost >= -0.01]
filas = validos.groupby([validos.org_id, validos.event_ts.dt.date, validos.service]).ngroups
print('Filas esperadas en org_daily_usage_by_service:', filas)

resource_id sin match en resources.csv: 0
org_id sin match en customers_orgs.csv: 0
Filas esperadas en org_daily_usage_by_service: 11050


## 6. Conclusiones

- La Master Data no tiene huérfanos, pero presenta nulos y valores fuera de rango (NPS, CSAT, fechas de login).
- Los eventos cambian de esquema el 18/07/2025; `carbon_kg` y `genai_tokens` solo existen en v2.
- `unit` se puede imputar desde `metric` porque la relación es uno a uno.
- Los costos menores a -0,01 van a quarantine y los picos se marcan con un flag.
- Cada archivo trae eventos de los 60 días, lo que obliga a un watermark amplio y a agregar en batch.